<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/RUL_%EC%98%88%EC%B8%A1_%EB%B0%8F_%EC%8B%9C%EA%B0%81%ED%99%94_%EC%BD%94%EB%93%9C_VTR_GPU_R00.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False


def generate_dummy_data(file_path):
    """비선형(가속화) 열화 패턴을 띄는 가상 데이터 생성"""
    print("Generating virtual accelerating degradation data...")
    np.random.seed(42)
    dates = pd.date_range(end=pd.Timestamp.now(), periods=1000, freq='1h')
    mean_values = []

    current_val = 100
    for i in range(1000):
        # 시간이 지날수록 상승폭이 커지는 비선형(Polynomial/Exponential) 패턴 모사
        step_increase = 0.2 + (i / 400)**2 + np.random.normal(0, 0.5)
        # 진동(노이즈)을 조금 더 강하게 추가 (스무딩 효과 테스트용)
        current_val += max(0, step_increase + np.random.normal(0, 1.0))
        mean_values.append(current_val)

    df = pd.DataFrame({'start_time': dates, 'mean': mean_values})
    df.to_parquet(file_path)
    print(f"Virtual data saved: {file_path}")


# ----------------------------------------
# PyTorch LSTM 모델 정의
# ----------------------------------------
class DegradationLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super(DegradationLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        # 마지막 타임스텝의 출력만 사용
        out = self.fc(out[:, -1, :])
        return out


def train_and_predict_lstm(df, threshold=1100, seq_length=24):
    """LSTM을 활용하여 변화량(Diff)을 학습하고 1100 도달 시점을 예측"""
    print("\n1. Preparing data for LSTM (Smoothing & Diff & Scaling)...")

    # [핵심 추가] 데이터 스무딩 (EMA: 지수 이동 평균 적용)
    # span 값(예: 24시간)이 클수록 더 부드러워지며, 데이터 주기에 맞게 튜닝하세요.
    ema_span = 24
    df['smoothed_mean'] = df['mean'].ewm(span=ema_span, adjust=False).mean()

    # 절대값이 아닌 '스무딩된 데이터'의 상승량(Diff)을 계산합니다.
    df['diff'] = df['smoothed_mean'].diff().fillna(0)

    # 변화량 스케일링
    scaler = MinMaxScaler()
    scaled_diff = scaler.fit_transform(df[['diff']].values)

    # 시퀀스 데이터 생성 (과거 seq_length 만큼의 패턴으로 다음 1스텝을 예측)
    X, y = [], []
    for i in range(len(scaled_diff) - seq_length):
        X.append(scaled_diff[i : i + seq_length])
        y.append(scaled_diff[i + seq_length])

    X = torch.FloatTensor(np.array(X))
    y = torch.FloatTensor(np.array(y))

    # GPU 디바이스 설정
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"2. Training LSTM on device: {device} 🚀")

    model = DegradationLSTM().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    dataset = TensorDataset(X, y)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    # 모델 학습
    epochs = 50
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch+1) % 10 == 0:
            print(f" - Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.6f}")

    print("3. Autoregressive Forecasting to Threshold...")
    model.eval()

    # 현재 데이터의 마지막 시퀀스 추출
    current_seq = scaled_diff[-seq_length:].reshape(1, seq_length, 1)
    current_seq_tensor = torch.FloatTensor(current_seq).to(device)

    # 미래 예측의 기준점도 노이즈가 있는 원본이 아닌 '스무딩된 마지막 값'을 사용합니다.
    last_mean = df['smoothed_mean'].iloc[-1]
    last_time = df['start_time'].iloc[-1]

    # 시간 간격(Frequency) 계산
    time_diff = df['start_time'].diff().mode()[0]

    predicted_means = []
    predicted_times = []

    current_mean = last_mean
    current_time = last_time
    steps = 0

    # 1100에 도달할 때까지 1스텝씩 재귀적으로 미래 예측
    with torch.no_grad():
        while current_mean < threshold:
            # 1. 다음 스텝의 상승량(Diff) 예측
            pred_scaled_diff = model(current_seq_tensor).cpu().numpy()
            pred_diff = scaler.inverse_transform(pred_scaled_diff)[0][0]

            # (안전장치) 모델이 음수를 예측할 경우 0 이상의 최소 상승분 보장
            pred_diff = max(0.01, pred_diff)

            # 2. 현재 mean 값에 상승량을 더함
            current_mean += pred_diff
            current_time += time_diff

            predicted_means.append(current_mean)
            predicted_times.append(current_time)

            # 3. 시퀀스 윈도우 한 칸 밀기 (다음 예측을 위해)
            new_seq = current_seq_tensor.cpu().numpy()
            new_seq = np.append(new_seq[:, 1:, :], [pred_scaled_diff], axis=1)
            current_seq_tensor = torch.FloatTensor(new_seq).to(device)

            steps += 1
            if steps > 10000: # 무한 루프 방지
                print("Warning: Prediction limit reached.")
                break

    print("\n==================================================")
    print(f" 🎯 LSTM 1100 도달 예상 시점 : {current_time.strftime('%Y년 %m월 %d일 %H시 %M분 %S초')}")
    print("==================================================\n")

    return predicted_times, predicted_means, current_time


def visualize_lstm_prediction(df, lstm_times, lstm_means, lstm_pred_time, threshold=1100):
    """LSTM 예측 결과 시각화"""
    print("4. Generating Visualization Chart...")
    plt.figure(figsize=(12, 7))

    # 1. 실제 원본 과거 데이터 (노이즈가 있는 상태를 흐리게 표시)
    plt.plot(df['start_time'], df['mean'], color='gray', alpha=0.3, linewidth=1.5, label='Actual Data (Raw Noise)')

    # 2. 스무딩된 과거 데이터 (EMA 곡선)
    plt.plot(df['start_time'], df['smoothed_mean'], color='black', linewidth=2, label='Smoothed Data (EMA)')

    # 3. PyTorch LSTM 예측 곡선
    plt.plot(lstm_times, lstm_means, color='#ff7f0e', linestyle='-', linewidth=2.5,
             label='LSTM Prediction (Non-linear)')
    plt.scatter([lstm_pred_time], [threshold], color='#ff7f0e', s=120, zorder=5)

    # 임계치 라인
    plt.axhline(y=threshold, color='red', linestyle=':', linewidth=2, label=f'Threshold ({threshold})')

    # 텍스트 주석 (LSTM)
    plt.annotate(f'Predicted Hit:\n{lstm_pred_time.strftime("%Y-%m-%d %H:%M")}',
                 xy=(lstm_pred_time, threshold), xytext=(-80, 30), textcoords='offset points',
                 arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=.2", color='#ff7f0e'),
                 fontsize=12, fontweight='bold', color='#d62728')

    plt.title('Prediction of Equipment RUL using Deep Learning (LSTM)', fontsize=16, fontweight='bold')
    plt.xlabel('Date / Time', fontsize=12)
    plt.ylabel('Sensor Value (mean)', fontsize=12)

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=12, loc='upper left')
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    parquet_file = 'sample_sensor_data_dl.parquet'

    # 가상의 가속화 패턴 데이터 생성
    if not os.path.exists(parquet_file):
        generate_dummy_data(parquet_file)

    try:
        df = pd.read_parquet(parquet_file)
        df['start_time'] = pd.to_datetime(df['start_time'])
        df = df.sort_values('start_time').reset_index(drop=True)

        # LSTM 단일 예측
        lstm_t, lstm_m, lstm_pred_time = train_and_predict_lstm(df, threshold=1100)

        # 차트 출력
        visualize_lstm_prediction(df, lstm_t, lstm_m, lstm_pred_time)

    except Exception as e:
        print(f"Error occurred: {e}")